# Pruning em CNNs — MIO-TCD (Kaggle)

**Uso opcional.** Este notebook roda o pipeline na GPU gratuita do Kaggle. Foi assim que o Leon rodou os Exp 1, 5 e 6. O Gabriel usa o próprio computador por padrão; use este notebook só se quiser ou precisar de GPU (ver `docs/KAGGLE.md`).

Requisitos: **Accelerator = GPU**, **Internet = On**, dataset `leonloureno/mio-tcd-subset-46k` adicionado em *Input* (compartilhado pelo Leon) e o secret `GH_TOKEN` (token do GitHub só com leitura deste repositório) ativado em *Add-ons → Secrets*.

Protocolo: `docs/SPEC.md`. Altere só `SEEDS` abaixo (ex.: `[1, 2]` roda as duas sementes em sequência). Para os Exp 2–4, troque as células do Exp 1/5/6 pelo script da poda L1 e adicione o checkpoint do baseline como *Input*.

Para rodar só a poda sem treinar o baseline de novo: adicione como *Input* a saída da versão que treinou o baseline e use `RUN_BASELINE = False`.

In [ ]:
SEEDS = [0]  # sementes a rodar em sequencia, ex.: [1, 2]
RUN_BASELINE = True  # False: reaproveita o checkpoint do baseline de uma versao anterior adicionada como Input

import glob, os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

REPO = "/kaggle/working/cnn-pruning-miotcd"
tok = UserSecretsClient().get_secret("GH_TOKEN")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", f"https://{tok}@github.com/LeonLourenco/cnn-pruning-miotcd.git", REPO], check=True)
# remove o token do .git/config: a pasta /kaggle/working vira saida publica da versao
subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/LeonLourenco/cnn-pruning-miotcd.git"], check=True)
del tok
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q torch-pruning==1.6.0

# localiza a pasta do dataset (o caminho de montagem varia entre versoes do Kaggle)
cands = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/car", recursive=True)]
assert cands, "dataset mio-tcd-subset-46k nao encontrado em /kaggle/input"
IMG_ROOT = cands[0]
print("imagens em", IMG_ROOT, "| classes:", len(os.listdir(IMG_ROOT)))

# aponta o config para o Kaggle (mudanca so nesta sessao, nao commitar)
!sed -i "s#img_root: .*#img_root: {IMG_ROOT}#" configs/default.yaml
!sed -i "s#num_workers: 2#num_workers: 4#" configs/default.yaml
!grep -n "img_root\|num_workers" configs/default.yaml

# auditoria dos dados: aborta se o split ou as imagens tiverem problema
import sys
r = subprocess.run([sys.executable, "scripts/audit_data.py", "--img-root", IMG_ROOT])
assert r.returncode == 0, "auditoria dos dados falhou: nao treine"

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA")
assert torch.cuda.is_available(), "ative Accelerator = GPU"

## Exp 1 — baseline denso

In [ ]:
def run(*args):
    """Roda um script do repositorio e aborta o notebook se ele falhar."""
    subprocess.run([sys.executable, *args], check=True)

for SEED in SEEDS:
    if RUN_BASELINE:
        run("-m", "src.train", "--seed", str(SEED))
    else:
        found = glob.glob(f"/kaggle/input/**/exp1_baseline_s{SEED}/exp1_baseline_s{SEED}.pt", recursive=True)
        assert found, "adicione como Input a saida da versao que treinou o baseline"
        shutil.copytree(os.path.dirname(found[0]), f"results/exp1_baseline_s{SEED}", dirs_exist_ok=True)
        print("baseline reaproveitado de", os.path.dirname(found[0]))

## Exp 5 e 6 — poda estruturada (máscara e física)

In [ ]:
for SEED in SEEDS:
    run("scripts/run_structured.py", "--seed", str(SEED))

## Resumo e saída

`results_s<sementes>.zip` aparece na aba *Output*: baixe e extraia na raiz do repositório local.
`baseline_s<semente>.zip` são os checkpoints do baseline para compartilhar com o Gabriel.

In [ ]:
run("scripts/aggregate_results.py")
for SEED in SEEDS:
    run("scripts/analyze_results.py", "--seed", str(SEED))
    subprocess.run(f"cd results && zip -q -r /kaggle/working/baseline_s{SEED}.zip exp1_baseline_s{SEED}", shell=True, check=True)
tag = "".join(str(s) for s in SEEDS)
subprocess.run(f"zip -q -r /kaggle/working/results_s{tag}.zip results", shell=True, check=True)
!ls -la /kaggle/working/*.zip